# Notebook 6.4: GIẢI PHẪU CHI TIẾT CẤU TRÚC ĐA PHƯƠNG THỨC (WSI + GENOMICS)
--- 
## MỤC ĐÍCH CỦA NOTEBOOK NÀY
Notebook này **không** được dùng để Training. Nó là một **"Bản vẽ kỹ thuật" (Blueprint)** siêu chi tiết, bóc tách từng dòng code, từng Tensor và từng thuật toán trong quy trình Hợp nhất Đa phương thức (Multimodal Late Fusion). 

Mục đích của nó là giúp bạn **chuẩn bị cho vòng bảo vệ đồ án/luận văn**: thấu hiểu tận gốc rễ những gì mô hình đang làm đằng sau bức màn đen (Blackbox), để tự tin trả lời các câu hỏi phản biện từ Hội đồng chấm thi.

### 🔍 Chúng ta sẽ đi qua các phần:
1. **Giải phẫu Dữ liệu đầu vào**: Hiểu bản chất File Genomics (500 gen) và File Hình ảnh WSI (.pt)
2. **Bí ẩn của Zero-Padding**: Giải quyết bài toán "kích thước ảnh không đồng đều" bằng `pad_collate`.
3. **Bóc tách Nhánh Thị Giác (Vision Encoder)**: Tại sao lại dùng TransMIL và `[CLS]` token?
4. **Bóc tách Nhánh Sinh Học (Genomics Encoder)**: Tại sao dùng MLP, LayerNorm và ép về không gian 512 chiều?
5. **Cơ chế Late Fusion (Hợp nhất Trễ)**: Triết lý đằng sau cú `torch.cat` thần thánh.
6. **Dummy Forward Pass**: Chạy chay 1 Batch để theo dõi đường đi của Tensor.
7. **Phụ lục Q&A**: Bộ câu hỏi "chống trượt" bảo vệ đồ án.

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# Cấu hình Font Tiếng Việt quốc dân cho Matplotlib (chống lỗi ô vuông trên Kaggle/Windows)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Tahoma', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✓ Đã nạp thành công các thư viện cơ bản.")

✓ Đã nạp thành công các thư viện cơ bản.


## 1. GIẢI PHẪU DỮ LIỆU ĐẦU VÀO (INPUT DATA)
Mô hình Multimodal yêu cầu 2 đầu vào hoàn toàn khác biệt nhau về mặt không gian vật lý:
- **Genomics (Dữ liệu bảng - 1D):** Mức độ biểu hiện (Expression Level) của 500 gen sinh ung thư.
- **WSI (Dữ liệu không gian - 2D):** Các bản vá hình ảnh mô học (Patches), đã được trích xuất đặc trưng bởi ResNet50 thành các vector 2048 chiều.

In [2]:
import os

# ---------------------------------------------------------
# 1.1 ĐỌC DỮ LIỆU GENOMICS ĐÃ TIỀN XỬ LÝ TỪ NB 6.0
# ---------------------------------------------------------
genomics_path = '/kaggle/input/datasets/trihuynhviprovcl/tcga-brca/tcga_brca_rna_500_scaled.csv'
# Nếu đang test dưới Local, ta có thể sinh dữ liệu ảo (dummy data) để mô phỏng nếu chưa có file CSV.
if os.path.exists(genomics_path):
    print(f"Đang đọc file gen tại: {genomics_path}")
    df_gen = pd.read_csv(genomics_path, index_col=0)
else:
    print(f"⚠️ Không tìm thấy file thật. Bật chế độ giả lập (Dummy Data) cho 5 bệnh nhân...")
    dummy_pids = ['TCGA-A1-A0SA', 'TCGA-B2-A1G2', 'TCGA-C3-A2H5', 'TCGA-D4-A3I8', 'TCGA-E5-A4J1']
    df_gen = pd.DataFrame(np.random.randn(5, 500), index=dummy_pids, columns=[f"GENE_{i}" for i in range(1, 501)])

print("\n🔍 KHÁM NGHIỆM MA TRẬN GENOMICS:")
print(f"- Kích thước (Shape): {df_gen.shape} -> [Số lượng Bệnh nhân, Số lượng Gen]")
print("- Giải thích: Mỗi bệnh nhân được đại diện bằng một vector 500 chiều (500 con số). Các số này đã được chuẩn hóa Z-Score (Mean=0, Std=1) để tránh việc một vài gen có giá trị bùng nổ (Outliers) lấn át các gen khác.")
display(df_gen.head(3))

# ---------------------------------------------------------
# 1.2 ĐỌC TENSOR WSI TỪ MỘT BỆNH NHÂN (VÍ DỤ)
# ---------------------------------------------------------
print("\n🔍 KHÁM NGHIỆM TENSOR ẢNH WSI (Whole Slide Image):")
# Giả lập tensor WSI của 1 bệnh nhân có 850 mảnh (patches)
dummy_wsi = torch.randn(850, 2048) 
print(f"- Kích thước Tensor WSI: {dummy_wsi.shape} -> [Số lượng Patch, Chiều Đặc trưng]")
print("- Cột 850: Nghĩa là tấm tiêu bản của bệnh nhân này được cắt nhỏ thành 850 bức ảnh con (patches).")
print("- Cột 2048: Nghĩa là mỗi bức ảnh con đã được nhét qua mạng ResNet50 (pre-trained ImageNet), lấy ở lớp cuối cùng (Flatten Layer) để sinh ra một dãy mã hóa dài 2048 con số.")

⚠️ Không tìm thấy file thật. Bật chế độ giả lập (Dummy Data) cho 5 bệnh nhân...

🔍 KHÁM NGHIỆM MA TRẬN GENOMICS:
- Kích thước (Shape): (5, 500) -> [Số lượng Bệnh nhân, Số lượng Gen]
- Giải thích: Mỗi bệnh nhân được đại diện bằng một vector 500 chiều (500 con số). Các số này đã được chuẩn hóa Z-Score (Mean=0, Std=1) để tránh việc một vài gen có giá trị bùng nổ (Outliers) lấn át các gen khác.


,GENE_1,GENE_2,GENE_3,GENE_4,GENE_5,GENE_6,GENE_7,GENE_8,GENE_9,GENE_10,...,GENE_491,GENE_492,GENE_493,GENE_494,GENE_495,GENE_496,GENE_497,GENE_498,GENE_499,GENE_500
TCGA-A1-A0SA,-0.185679,0.731185,-0.544498,-1.606858,0.551598,-0.983085,0.034457,-1.593454,-1.621979,-0.763229,...,0.302755,2.737658,-0.289620,-0.020156,0.895267,0.991720,1.104388,-0.846085,-0.101858,-1.704865
TCGA-B2-A1G2,0.221601,-0.174721,-0.159620,-1.785108,1.979742,-2.850817,-0.694340,-1.947827,-0.487809,-0.381064,...,-1.301076,-1.189661,0.149827,0.981347,-1.048602,0.582751,-1.565128,-0.387728,0.574587,0.225798
TCGA-C3-A2H5,1.959220,-1.556434,-0.572472,0.355228,1.511908,1.354585,0.970415,-0.610069,1.634955,0.573754,...,2.629009,1.187889,-0.764806,-0.569642,-0.222669,1.912211,0.276162,1.308317,-0.279210,0.646134



🔍 KHÁM NGHIỆM TENSOR ẢNH WSI (Whole Slide Image):
- Kích thước Tensor WSI: torch.Size([850, 2048]) -> [Số lượng Patch, Chiều Đặc trưng]
- Cột 850: Nghĩa là tấm tiêu bản của bệnh nhân này được cắt nhỏ thành 850 bức ảnh con (patches).
- Cột 2048: Nghĩa là mỗi bức ảnh con đã được nhét qua mạng ResNet50 (pre-trained ImageNet), lấy ở lớp cuối cùng (Flatten Layer) để sinh ra một dãy mã hóa dài 2048 con số.


<div class="alert alert-warning">
<strong>Giáo viên hỏi:</strong> <i>"Tại sao số lượng patch (850) của mỗi bệnh nhân lại khác nhau? Làm sao đưa vào mạng huấn luyện được khi chúng nó lệch chiều?"</i><br>
<strong>Bạn trả lời:</strong> <i>"Dạ thưa cô/thầy, kích thước khối u của mỗi người là to nhỏ khác nhau, người ung thư nặng mô rải rác nhiều sẽ cắt được hàng ngàn patch, người khối u nhỏ chỉ cắt được vài chục patch. Để huấn luyện theo Batch (nhiều bệnh nhân cùng lúc) trên GPU, ta không thể nối các tensor lệch chiều. Nên em đã dùng kỹ thuật <strong>Zero-Padding (Đệm số 0)</strong> thông qua hàm <code>pad_collate</code> để bù độ dài cho bằng với người có số patch dài nhất trong Batch đó ạ."</i>
</div>

## 2. BÍ ẨN CỦA ZERO-PADDING (`pad_collate`)
Hãy xem đoạn code bóc tách quá trình này hoạt động ra sao!

In [3]:
# Code mô phỏng pad_collate (Giống hệt code trong hàm collate_fn của DataLoader)
def pad_collate_demo(batch_wsi_tensors):
    """
    batch_wsi_tensors: List chứa các tensor WSI của nhiều bệnh nhân trong 1 batch
    """
    # 1. Tìm ra bệnh nhân có nhiều patch nhất trong nhóm (Max Length)
    lengths = [tensor.size(0) for tensor in batch_wsi_tensors]
    max_len = max(lengths)
    
    padded_tensors = []
    for tensor in batch_wsi_tensors:
        pad_size = max_len - tensor.size(0)
        # Nếu thiếu bao nhiêu patch, đệm bằng ma trận toàn số 0 (zeros) kích thước [pad_size, 2048]
        if pad_size > 0:
            zero_padding = torch.zeros(pad_size, 2048)
            padded = torch.cat([tensor, zero_padding], dim=0)
        else:
            padded = tensor
        padded_tensors.append(padded)
    
    # 2. Ghép lại thành 1 Tensor 3D duy nhất để đẩy lên GPU
    batch_tensor = torch.stack(padded_tensors, dim=0)
    return batch_tensor, lengths

# --- TEST THỬ ---
patient_A = torch.randn(100, 2048)  # U nhỏ (100 patch)
patient_B = torch.randn(450, 2048)  # U to trung bình (450 patch)
patient_C = torch.randn(50, 2048)   # U rất nhỏ (50 patch)

print("Tình trạng Tensor TRƯỚC KHI PAD:")
print(f"- Bệnh nhân A: {patient_A.shape}")
print(f"- Bệnh nhân B: {patient_B.shape}")
print(f"- Bệnh nhân C: {patient_C.shape}")
print("-> GPU KHÔNG THỂ HỢP NHẤT (STACK) CHÚNG LẠI!")

batch_tensor, original_lengths = pad_collate_demo([patient_A, patient_B, patient_C])

print("\nTình trạng Tensor SAU KHI PAD:")
print(f"- Kích thước Batch Tensor (GPU Ready): {batch_tensor.shape} -> [Batch_size=3, Max_Patches=450, Dim=2048]")
print("-> Bệnh nhân A đã được đệm thêm 350 hàng số 0 ở đuôi.")
print("-> Bệnh nhân C đã được đệm thêm 400 hàng số 0 ở đuôi.")

Tình trạng Tensor TRƯỚC KHI PAD:
- Bệnh nhân A: torch.Size([100, 2048])
- Bệnh nhân B: torch.Size([450, 2048])
- Bệnh nhân C: torch.Size([50, 2048])
-> GPU KHÔNG THỂ HỢP NHẤT (STACK) CHÚNG LẠI!

Tình trạng Tensor SAU KHI PAD:
- Kích thước Batch Tensor (GPU Ready): torch.Size([3, 450, 2048]) -> [Batch_size=3, Max_Patches=450, Dim=2048]
-> Bệnh nhân A đã được đệm thêm 350 hàng số 0 ở đuôi.
-> Bệnh nhân C đã được đệm thêm 400 hàng số 0 ở đuôi.


## 3. BÓC TÁCH KIẾN TRÚC VISION ENCODER (TransMIL)

Đây là thuật toán phân tích Hình học (Morphology) cốt lõi của bài toán. 
WSI có hàng ngàn bản vá (patches), làm sao mô hình biết nên tập trung vào vùng nào chứa tế bào ung thư? 

**Cơ chế hoạt động:** 
1. **Linear Projection:** Ép chiều dữ liệu từ 2048 xuống 512.
2. **[CLS] Token:** Chèn thêm 1 vector đặc biệt vào đầu chuỗi. Giống như một "Người Lớp Trưởng". Ban đầu nó không biết gì, nhưng qua quá trình Self-Attention (Giao tiếp), nó sẽ "hỏi han" hàng ngàn bản vá bên dưới để gom nhặt thông tin xem toàn bộ mô này là ung thư nhóm nào.
3. **Transformer Layer:** Bộ não thật sự. Giúp các patch giao tiếp với nhau (Ví dụ patch Mạch máu nói chuyện với patch Tế bào hoại tử kế bên).
4. Cuối cùng, ta vứt hết các patch đi, **chỉ giữ lại duy nhất vector [CLS] (kích thước 512)**. Nó chính là kết tinh (Đại diện Tối cao) của cả tấm WSI.

In [4]:
class Vision_Encoder_Demo(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. Ép chiều: 2048 -> 512 (Để giảm thiểu độ phức tạp và tham số)
        self.fc1 = nn.Linear(2048, 512)
        
        # 2. Khởi tạo [CLS] Token. Tham số nn.Parameter nghĩa là nó có thể học (Learnable) trong lúc train.
        self.cls_token = nn.Parameter(torch.randn(1, 1, 512))
        
        # 3. Lớp Attention ảo (Dùng lớp Transformer_Encoder đơn giản của PyTorch để minh họa)
        encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=512, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        # 4. LayerNorm chuẩn hóa cuối cùng
        self.norm = nn.LayerNorm(512)

    def forward(self, x):
        # In ra quá trình biến đổi Shape
        print(f"[Vision] 1. Input WSI Shape : {x.shape} -> [Batch, Patches, 2048]")
        
        x = self.fc1(x)
        print(f"[Vision] 2. Sau FC1 (Ép chiều): {x.shape} -> [Batch, Patches, 512]")
        
        B = x.shape[0]  # Lấy Batch Size (Số lượng bệnh nhân)
        # Nhân bản [CLS] token cho bằng số lượng bệnh nhân trong Batch
        cls_tokens = self.cls_token.expand(B, -1, -1) 
        
        # Ghép [CLS] token vào ĐẦU của chuỗi Patches
        x = torch.cat((cls_tokens, x), dim=1)
        print(f"[Vision] 3. Sau khi chèn CLS : {x.shape} -> (Patches + 1) -> Vị trí 0 là CLS.")
        
        # Chạy qua Transformer (Self-Attention)
        x = self.transformer(x)
        
        # Lấy duy nhất vị trí 0 (Tức là [CLS] token) làm đại diện
        cls_out = x[:, 0, :]  
        cls_out = self.norm(cls_out)
        print(f"[Vision] 4. Bóc tách [CLS]   : {cls_out.shape} -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA ẢNH WSI.")
        return cls_out

# Chạy thử với Batch WSI lúc nãy
vision_model = Vision_Encoder_Demo()
vision_features_extracted = vision_model(batch_tensor)
print("\n=> Vision Nhánh Đã Hoàn Tất Xử Lý!")

[Vision] 1. Input WSI Shape : torch.Size([3, 450, 2048]) -> [Batch, Patches, 2048]
[Vision] 2. Sau FC1 (Ép chiều): torch.Size([3, 450, 512]) -> [Batch, Patches, 512]
[Vision] 3. Sau khi chèn CLS : torch.Size([3, 451, 512]) -> (Patches + 1) -> Vị trí 0 là CLS.
[Vision] 4. Bóc tách [CLS]   : torch.Size([3, 512]) -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA ẢNH WSI.

=> Vision Nhánh Đã Hoàn Tất Xử Lý!


## 4. BÓC TÁCH KIẾN TRÚC GENOMICS ENCODER (Tabular MLP)

Nhánh Gen có đầu vào siêu đơn giản: Chỉ 500 con số (1D Array). Do đó ta không cần Transformer, chỉ cần mạng MLP (Multi-Layer Perceptron) truyền thống.

**Quy trình biến đổi (Forward Pass):**
1. `Linear(500, 256)`: Bóp nghẹt thông tin (Bottleneck) để ép mô hình chắt lọc các gen cốt lõi sinh ung thư.
2. `LayerNorm(256)`: Chuẩn hóa, kéo phân phối dữ liệu về chuẩn. (Giúp ổn định Gradient). Dùng LayerNorm thay BatchNorm vì BatchNorm sẽ sập nguồn (Crash) nếu `BatchSize = 1 hoặc 2` trong môi trường Đa GPU (DataParallel).
3. `LeakyReLU`: Hàm kích hoạt phi tuyến (Non-linear).
4. `Dropout(0.3)`: Bắn bỏ ngẫu nhiên 30% nơ-ron để chống Học vẹt (Overfitting).
5. `Linear(256, 512)`: Phóng to không gian lên 512 chiều. 
   - *Tại sao lại là 512?* -> Để bằng đúng với số chiều 512 của ảnh WSI ([CLS] token). Cân bằng Cán cân Quyền lực giữa Hình ảnh và Sinh học khi Hợp nhất!

In [5]:
class Genomics_Encoder_Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(500, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.LeakyReLU(0.1)
        )
        
    def forward(self, x):
        print(f"[Genomics] 1. Input Gen Shape : {x.shape} -> [Batch, 500]")
        out = self.net(x)
        print(f"[Genomics] 2. Sau khi qua MLP : {out.shape} -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA GEN.")
        return out

# Sinh dữ liệu Gen giả lập (3 bệnh nhân, 500 gen)
batch_gen_tensor = torch.randn(3, 500)

gen_model = Genomics_Encoder_Demo()
gen_features_extracted = gen_model(batch_gen_tensor)
print("\n=> Genomics Nhánh Đã Hoàn Tất Xử Lý!")

[Genomics] 1. Input Gen Shape : torch.Size([3, 500]) -> [Batch, 500]
[Genomics] 2. Sau khi qua MLP : torch.Size([3, 512]) -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA GEN.

=> Genomics Nhánh Đã Hoàn Tất Xử Lý!


## 5. LATE FUSION (HỢP NHẤT TRỄ) & PHÂN LOẠI CUỐI (CLASSIFICATION)

<div class="alert alert-info">
<strong>Giáo viên hỏi:</strong> <i>"Em gọi đây là Đa phương thức (Multimodal), vậy tại sao em lại dùng Late Fusion (Hợp nhất trễ) ở giai đoạn cuối mà không nhập ma trận Gen vào thẳng ảnh ngay từ ban đầu (Early Fusion)?"</i><br>
<strong>Bạn trả lời:</strong> <i>"Dạ thưa cô/thầy, Hình ảnh WSI và Hồ sơ Gen biểu hiện (RNA-Seq) tồn tại ở hai không gian hoàn toàn không đồng nhất (Heterogeneous). Hình ảnh là không gian Tọa độ Không gian/Pixel (Spatial), còn Gen là dữ liệu Vô hướng/Phân tử 1D (Molecular). Nếu ghép nối thẳng hàng từ ban đầu (Early Fusion), mạng nơ-ron sẽ bị nhiễu loạn trầm trọng vì không cùng tỷ lệ và bản chất vật lý. Do đó, em phải để 2 nhánh chạy riêng biệt để tự chắt lọc ra Vector ngữ nghĩa bậc cao (512D) của riêng nó, rồi mới nối (Cat) chúng lại với nhau ở phút 89. Kỹ thuật này gọi là <strong>Deep Late Fusion</strong> ạ."</i>
</div>

In [6]:
class Multimodal_LateFusion_Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = Vision_Encoder_Demo()
        self.gen = Genomics_Encoder_Demo()
        
        # BỘ PHÂN LOẠI CUỐI (CLASSIFIER HEAD)
        # Đầu vào = 512 (Vision) + 512 (Gen) = 1024 chiều
        # Đầu ra = 4 chiều (Tương ứng xác suất của 4 loại ung thư PAM50: LumA, LumB, Basal, HER2)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.LayerNorm(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(256, 4) # 4 classes PAM50
        )
        
    def forward(self, wsi, rna):
        print("=========== BẮT ĐẦU CHU TRÌNH FORWARD PASS ===========")
        v_feat = self.vision(wsi)
        g_feat = self.gen(rna)
        
        print("\n--- TIẾN HÀNH LATE FUSION ---")
        # Dùng torch.cat để NỐI THEO CHIỀU NGANG (dim=1)
        fusion = torch.cat((v_feat, g_feat), dim=1)
        print(f"[Fusion] Sau khi torch.cat(vision, gen) : {fusion.shape} -> [Batch, 1024] (512 + 512)")
        
        print("\n--- RA QUYẾT ĐỊNH (CLASSIFICATION) ---")
        logits = self.classifier(fusion)
        print(f"[Output] Sau Classifier Head            : {logits.shape} -> [Batch, 4 Classes]")
        
        print("=========== KẾT THÚC CHU TRÌNH ===========")
        return logits

# ---------- THỰC THI (TEST FORWARD PASS) ----------
final_model = Multimodal_LateFusion_Demo()

# Đẩy dữ liệu Batch 3 bệnh nhân vào Mạng lưới Tối cao
predictions = final_model(wsi=batch_tensor, rna=batch_gen_tensor)

print("\n🎯 KẾT QUẢ DỰ ĐOÁN THÔ (LOGITS) CHO 3 BỆNH NHÂN:")
print(predictions)

# Quy đổi ra Xác suất (%)
probabilities = torch.softmax(predictions, dim=1)
print("\n🎯 KẾT QUẢ ĐÃ QUY ĐỔI RA XÁC SUẤT (%) CHO 4 NHÓM PAM50:")
print(torch.round(probabilities * 100))

=========== BẮT ĐẦU CHU TRÌNH FORWARD PASS ===========
[Vision] 1. Input WSI Shape : torch.Size([3, 450, 2048]) -> [Batch, Patches, 2048]
[Vision] 2. Sau FC1 (Ép chiều): torch.Size([3, 450, 512]) -> [Batch, Patches, 512]
[Vision] 3. Sau khi chèn CLS : torch.Size([3, 451, 512]) -> (Patches + 1) -> Vị trí 0 là CLS.
[Vision] 4. Bóc tách [CLS]   : torch.Size([3, 512]) -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA ẢNH WSI.
[Genomics] 1. Input Gen Shape : torch.Size([3, 500]) -> [Batch, 500]
[Genomics] 2. Sau khi qua MLP : torch.Size([3, 512]) -> [Batch, 512]. ĐÂY LÀ TINH HOA CỦA GEN.

--- TIẾN HÀNH LATE FUSION ---
[Fusion] Sau khi torch.cat(vision, gen) : torch.Size([3, 1024]) -> [Batch, 1024] (512 + 512)

--- RA QUYẾT ĐỊNH (CLASSIFICATION) ---
[Output] Sau Classifier Head            : torch.Size([3, 4]) -> [Batch, 4 Classes]
=========== KẾT THÚC CHU TRÌNH ===========

🎯 KẾT QUẢ DỰ ĐOÁN THÔ (LOGITS) CHO 3 BỆNH NHÂN:
tensor([[-0.5421, -0.2185, -0.3864, -0.0830],
        [-0.4562,  0.2155, -0.5342, -0

## 6. PHỤ LỤC: BỘ CÂU HỎI "CHỐNG TRƯỢT" KHI BẢO VỆ

Nếu gặp các Giảng viên chuyên về Toán, Deep Learning hay Y Sinh, đây là những câu "Trùm cuối" có thể họ sẽ hỏi. Hãy học thuộc ý chính để trả lời lưu loát!

**Q1. Kích thước mô hình của em lớn không? Chạy trên máy thường được không?**
> Trả lời: "Kích thước kiến trúc Late Fusion của em rất nhẹ (chỉ tốn khoảng vài triệu tham số). Nguyên nhân do em KHÔNG dùng ResNet50 để huấn luyện lại từ đầu (End-to-End Image). Em đã sử dụng ResNet50 đóng vai trò Trích xuất đặc trưng ngoại tuyến (Offline Feature Extractor), lưu sẵn ra file `.pt` (Feature Dimension 2048). Quá trình train chính chỉ huấn luyện các lớp MLP, Transformer Attention siêu nhẹ phía sau. Nhờ vậy, ngay cả Card T4 Kaggle cũng có thể chạy mượt mà mà không lo sập vRAM (OOM)."

**Q2. Em có biết cơ chế Attention (Self-Attention) trong nhánh Hình học hoạt động thế nào không?**
> Trả lời: "Self-Attention tính toán sự tương quan (Ma trận Q, K, V) giữa mọi mảnh cắt (patch) với nhau trên toàn bộ bức ảnh. Các bản vá có mô hình khối u tương tự nhau sẽ 'bỏ phiếu' cho nhau một trọng số Attention cao, giúp mô hình bắt được tính bất đồng nhất (Heterogeneity) lan tỏa trên toàn khối u thay vì chỉ nhìn cục bộ như CNN truyền thống."

**Q3. Có rủi ro nhánh Vision áp đảo hoàn toàn nhánh Gen khiến Gen trở nên vô dụng (Modality Collapse)?**
> Trả lời: "Đó chính là lý do em thiết kế Nhánh Gen và Nhánh WSI CÙNG ÉP VỀ 512 CHIỀU. Nếu nhánh Vision có 2048 chiều mà Gen chỉ có 128 chiều, khi nối lại, gradient sẽ chảy toàn bộ về Vision. Sự cân bằng vector 512-512 ép mô hình phải lắng nghe sinh học và hình ảnh một cách công bằng nhất (Equal Capacity). Sự nhảy vọt F1 lên 0.85 (Nb 6.2) là minh chứng rõ nhất cho việc cả 2 nhánh đều có hiệu lực."

**Q4. Vì sao lại dùng Variance Threshold (Chọn Top 500) thay vì ném cả 20,500 gen vào mô hình?**
> Trả lời: "Nếu ném cả 20,500 gen, mô hình sẽ dính 'Lời nguyền Số chiều' (Curse of Dimensionality). Hơn 90% số lượng gen trong đó là Gen giữ nhà (Housekeeping genes) hoặc các mảnh rác (Non-coding) biểu hiện giống hệt nhau ở mọi người. Variance Threshold cho phép em lọc ra 500 gen 'điên rồ nhất', dao động dữ dội nhất giữa các nhóm bệnh, đó chính là các Biomarker (Dấu ấn sinh học) thực thụ quyết định kiểu hình của khối u. Càng ít gen, mạng Neural mạng chạy càng nhanh và càng tránh được Overfitting."